# اليوم الرابع: فسّر النموذج واختبر احتمالاته
**SDA-DSC-211 · Tamweel Lite · Day 4 · 60 minutes**

إعداد وتقديم **ميعاد المري | Meaad Al-Marri**

ستسأل سؤالين مختلفين: ما الذي يستخدمه النموذج، وهل تعكس درجاته تكرار الحدث؟ تقارن أهمية الخصائص، وتفسّر طلبًا اصطناعيًا، وتختبر المعايرة على فترة منفصلة. ثم تفحص استقرار النتيجة وأثر منطقة المراجعة على السعة.

**تسليمك:** `INTERPRETABILITY_REPORT.md` مع رسوم SHAP والمعايرة وملفات الأدلة. افتح جلسة **CPU مجانية جديدة** واختر **Runtime → Run all**. لا تحتاج إلى نتائج يوم سابق أو ربط Drive أو اشتراك. بعد التشغيل، اكتب تفسيرك في خلية «اكتب تقريرك»، وأعدها وخلية التصدير.

| الوقت | مهمتك |
|---|---|
| 0–10 | افصل أدوار البيانات وافحص أهمية التبديل |
| 10–25 | اقرأ beeswarm ووحدة SHAP |
| 25–35 | فسّر طلبًا واحدًا وافحص ثبات أسبابه |
| 35–48 | قارن المعايرة وBrier وECE |
| 48–54 | اقرأ حدود الاستقرار وطابور المراجعة |
| 54–60 | أكمل تقريرك واحفظ أدلتك |

The data are fictional. Explanations describe a fitted model, not causes or real lending decisions. Calibration improvement is measured, never assumed.

## 1. جهّز بيئتك | Setup
يفحص الإعداد الإصدارات وبصمات الملفات. إذا طلب إعادة الجلسة، اختر Restart session ثم Run all. لا تتجاوز فحص البصمة. تستخدم الأدوات المثبتة نفسها، ومنها scikit-learn 1.6.1 وLightGBM 4.6.0 وSHAP 0.52.0.

نثبت النموذج والإعدادات قبل التقييم، ونستخدم `FrozenEstimator` لمعايرة نموذج مكتمل التدريب دون إعادة تدريبه على بيانات المعايرة. لا نستخدم تقسيمًا عشوائيًا افتراضيًا يخلط الزمن والعملاء.

In [ ]:
from pathlib import Path
import hashlib, importlib.util, os, runpy, sys, time, urllib.request, urllib.error

RUN_STARTED = time.perf_counter()
COURSE_REVISION = "fe0c0204e6076a7ac2139b7336485a097343fb8a"
MANIFEST_SHA256 = "86d15e813caf833b7d86512808176388d4123dc3c01c6525ee849a76e8322c59"
SETUP_SHA256 = "79608f8f9559d9e6eb09d51ee541ee12d7e9262ac57d1ca3a948c8ecb14f0f35"
IN_COLAB = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
ROOT = Path("/content/tamweel") if IN_COLAB else Path(os.environ.get("TAMWEEL_PROJECT_DIR", ".")).resolve()
setup_path = ROOT / "scripts/setup_colab.py"
setup_path.parent.mkdir(parents=True, exist_ok=True)
if not setup_path.exists() or hashlib.sha256(setup_path.read_bytes()).hexdigest() != SETUP_SHA256:
    setup_url = f"https://raw.githubusercontent.com/almiyead-rgb/sda-dsc-211-student-template/{COURSE_REVISION}/scripts/setup_colab.py"
    for attempt in range(3):
        try:
            with urllib.request.urlopen(setup_url, timeout=45) as response:
                setup_bytes = response.read(200000)
            if hashlib.sha256(setup_bytes).hexdigest() != SETUP_SHA256:
                raise ValueError("Setup checksum mismatch. Reopen the released notebook.")
            setup_path.write_bytes(setup_bytes)
            break
        except (urllib.error.URLError, TimeoutError):
            if attempt == 2:
                raise RuntimeError("Download unavailable. Retry setup or upload the unchanged project files; see READINESS_GUIDE.")
            time.sleep(attempt + 1)
environment = runpy.run_path(str(setup_path))["prepare"](ROOT, COURSE_REVISION, MANIFEST_SHA256)
FAST_MODE, FULL_MODE, SEED, N_JOBS = True, False, 211, 2

from setup_colab import fetch_verified
PREVIOUS_SUPPORT = {
    "scripts/day2_validation.py": ("1e1da4acbee8941bef07245b259fb6c27ced4ad7", "d2fd0353330c3629bdaed6c570528792955fce232540a85d33532a9432522f43"),
    "scripts/day3_decision.py": ("c2dc52e129878d0d487db1874891c658de7c4d0d", "40f176633c91fc530db222978bc44495a401b888b1ce448728948f7ae0ca9e12")}
for relative, (revision, digest) in PREVIOUS_SUPPORT.items():
    fetch_verified(f"https://raw.githubusercontent.com/almiyead-rgb/sda-dsc-211-student-template/{revision}/{relative}", ROOT / relative, digest)
LAB_REVISION = "4f6892d5c02923b5f8e3c78f4fcead73b043570f"
SUPPORT_HASHES = {'scripts/day4_trust.py': '7e0f54bdbed94b28d09cddd02ce0ba0471d5046f05c7b70589e3e4be9902f20c', 'data/day4_shap_example.npz': 'de6eefb4f0b1dfd54d4a71e2b73cf8383cea6386e661f8af8662d478cd957e2b', 'data/day4_shap_example.json': '8169c70dc327a7ffdff748c365d71544fe951f3b196974e33f09dccb580a8122'}
for relative, digest in SUPPORT_HASHES.items():
    fetch_verified(f"https://raw.githubusercontent.com/almiyead-rgb/sda-dsc-211-student-template/{LAB_REVISION}/{relative}", ROOT / relative, digest)
print("Day 4 setup verified | إعداد اليوم الرابع جاهز")

## 2. افصل الأدوار قبل التدريب | Data roles
الهدف حدث تعثر اصطناعي **خلال 90 يومًا بعد الطلب**. تقسم الطلبات إلى أربعة أدوار، وتحجز عملاء الدور الأحدث أولًا ثم تستبعدهم من الأدوار الأقدم. المعرّفات والتاريخ لا يدخلان النموذج، وكل نتائج الدور السابق تنضج قبل بداية التالي.

| الدور | الفترة أو الشرط | الاستخدام الوحيد |
|---|---|---|
| fit | تاريخ الطلب +90 يومًا < 2023-07-01 | تعلم التعويض والنموذج الموزون |
| calibration | يوليو–سبتمبر 2023 | تعلم تحويل sigmoid فقط |
| policy | يناير–مارس 2024 | اختيار العتبة بخسارة 10×FN+FP وسقف 12% |
| evaluation | يوليو–ديسمبر 2024 | القياس والتفسير بعد التثبيت |

تُستبعد الفجوات وصفوف العملاء المتعارضين ولا تُنقل إلى التدريب. **فترة التقييم منفصلة داخل هذا التطبيق، لكنها سبق ظهورها في تمارين الدورة؛ ليست اختبارًا نهائيًا لم يمسّ.** لا يستخدم ملف التحدي للنمذجة اليوم. الفترات صغيرة نسبيًا، خصوصًا موجبات المعايرة؛ اقرأ عددها بجانب النتيجة.

FAST: 80 شجرة، 300 صف SHAP، 3 تبديلات و200 تكرار bootstrap. FULL اختياري: 160 شجرة، 600 صف، 5 تبديلات و500 تكرار. منطقة المراجعة ±0.02 احتمال تُحدد قبل التقييم، وهي سيناريو تشغيلي وليست فترة ثقة.

In [ ]:
import json, platform, zipfile
from dataclasses import asdict
import numpy as np
import pandas as pd
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
from scipy.special import expit
from IPython.display import display
from data_checks import validate_frame
from day2_validation import deduplicate_applications
from day4_trust import (TARGET, ROLES, TrustConfig, split_roles, fit_and_calibrate,
    probability_metrics, reliability_table, permutation_report, explain_sample, load_shap_example,
    ShapBudgetExpired, reason_codes, local_stability, cluster_bootstrap, choose_policy_threshold,
    transport_threshold, map_probability, review_audit, reflection_check)

contract = json.loads((ROOT / "data/data_contract.json").read_text(encoding="utf-8"))
train, duplicates = deduplicate_applications(pd.read_csv(ROOT / "data/tamweel_train.csv"))
validate_frame(train, contract, "train")
FAST_MODE, FULL_MODE = True, False
assert FAST_MODE != FULL_MODE
config = TrustConfig() if FAST_MODE else TrustConfig(trees=160, shap_rows=600, permutation_repeats=5, bootstrap_repeats=500)
USE_SHAP_EXAMPLE = False  # Discussion only; not a personal submission.
ASSESSMENT_MODE = os.environ.get("ASSESSMENT_MODE", "0") == "1"
ARTIFACTS, REPORTS = ROOT / "artifacts", ROOT / "reports"
ARTIFACTS.mkdir(exist_ok=True); REPORTS.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
parts, assignments = split_roles(train)
role_summary = pd.DataFrame([{"role": r, "rows": len(p), "customers": p.customer_id.nunique(),
    "positives": int(p[TARGET].sum()), "positive_rate": p[TARGET].mean(),
    "first_date": p.application_date.min(), "last_date": p.application_date.max()} for r, p in parts.items()])
display(role_summary)
print("Excluded gaps/customer overlaps:", int((assignments.role == "excluded_gap_or_customer").sum()))
model, calibrated_model, predictions, provenance = fit_and_calibrate(parts, contract, config)
evaluation = predictions[predictions.role == "evaluation"].copy()
policy_rows = predictions[predictions.role == "policy"].copy()
chosen_policy, policy_sweep = choose_policy_threshold(policy_rows)
mapping = provenance["calibration_mapping"]
calibrated_threshold = transport_threshold(chosen_policy["threshold"], mapping)
print("Model, sigmoid and policy threshold frozen before evaluation.")

## 3. ما الذي يستخدمه النموذج؟ | Permutation importance
نحسب انخفاض **Average Precision** عند تبديل خاصية في بيانات التقييم، دون إعادة تدريب. نحافظ على إشارات المنطقة الثلاث معًا عند التبديل كي لا نصنع تركيبات إقليمية مستحيلة. رسم gain يصف مكاسب تقسيمات التدريب؛ رسم التبديل يصف اعتماد النموذج على المعلومات في هذه العينة.

الانخفاض قد يكون سالبًا؛ لا نخفيه. أشرطة التشتت هي انحراف بين التبديلات وليست فترة ثقة. الترابط بين الخصائص قد يوزع الإشارة بينها ويقلل أهمية تبديل كل واحدة منفردة؛ لا تختار خصائص جديدة ثم تسمي التقييم نفسه اختبارًا مستقلًا.

In [ ]:
permutation = permutation_report(model, parts["evaluation"], contract, config)
display(permutation[["feature_group", "mean_ap_drop", "repeat_sd", "training_gain"]].head(10).round(4))
fig, axes = plt.subplots(1, 2, figsize=(13, 5), layout="constrained")
for ax, value, title in [(axes[0], "training_gain", "Training split gain"), (axes[1], "mean_ap_drop", "Held-out AP drop")]:
    top = permutation.nlargest(10, value).sort_values(value)
    ax.barh(top.feature_group, top[value], color="#2b6b98",
            xerr=top.repeat_sd if value == "mean_ap_drop" else None, capsize=3)
    ax.axvline(0, color="#777", lw=.8); ax.set_title(title)
    ax.set_xlabel("Gain units" if value == "training_gain" else "Average Precision decrease; whiskers = repeat SD")
fig.suptitle(f"LIVE · {len(evaluation):,} evaluation requests · correlated features can share signal", fontsize=12)
fig.savefig(ARTIFACTS / "permutation_importance.png", bbox_inches="tight"); plt.show()

## 4. اقرأ التفسير العام ووحدته | SHAP beeswarm
نختار عينة عشوائية ثابتة من التقييم، ثم نفسر النموذج الخام الموزون بـ TreeSHAP. الخلفية هي أعداد مسارات أشجار التدريب (`tree_path_dependent`). يتعلم التعويض من fit فقط؛ قيم اللون هي المدخلات المستخدمة بعد التعويض.

**raw_margin = base_value + مجموع SHAP**، ثم **raw_probability = sigmoid(raw_margin)**. الوحدة هنا **log-odds**، وليست نقاطًا مئوية. لا تحوّل كل مساهمة منفردة إلى احتمال ثم تجمعها. كذلك sigmoid(base_value) ليس بالضرورة متوسط الاحتمالات. هذا الرسم لا يفسر نموذج المعايرة المركب.

رتّب قراءتك: متوسط القيمة المطلقة يحدد ترتيب الخصائص، والاتجاه يرفع الدرجة الخام أو يخفضها، واللون يوضح قيمة الخاصية، والانتشار يصف اختلاف الأثر بين الطلبات. هذا تفسير للنموذج وليس إثبات سببية أو عدالة.

In [ ]:
try:
    if USE_SHAP_EXAMPLE:
        raise ShapBudgetExpired("Educational example requested.")
    shap_arrays, shap_metadata = explain_sample(model, parts["evaluation"], contract, config)
except ShapBudgetExpired:
    shap_arrays, shap_metadata = load_shap_example(ROOT / "data/day4_shap_example.npz",
        ROOT / "data/day4_shap_example.json", model, parts["evaluation"], contract, ASSESSMENT_MODE)

import shap
EXPLANATION_SOURCE = shap_metadata["source"]
if EXPLANATION_SOURCE != "LIVE":
    print("EDUCATIONAL_EXAMPLE — مثال موسوم للمناقشة، غير صالح للتسليم كتجربتك. أعد SHAP لاحقًا.")
explanation = shap.Explanation(values=shap_arrays["values"], base_values=shap_arrays["base_values"],
    data=shap_arrays["data"], feature_names=list(shap_arrays["feature_names"]))
shap_global = pd.DataFrame({"feature": shap_arrays["feature_names"],
    "mean_absolute_shap_log_odds": np.abs(shap_arrays["values"]).mean(axis=0)}).sort_values("mean_absolute_shap_log_odds", ascending=False)
display(shap_global.head(10).round(4))
plt.figure()
shap.plots.beeswarm(explanation, max_display=12, show=False, plot_size=(11, 6.5))
plt.xlabel("SHAP contribution to raw log-odds (not probability points)")
plt.title(f"{EXPLANATION_SOURCE} · {len(shap_arrays['application_ids'])} sampled evaluation requests", fontsize=12)
plt.savefig(ARTIFACTS / "shap_beeswarm.png", bbox_inches="tight"); plt.show()
print("Maximum additivity error:", shap_metadata["max_additivity_error"])

## 5. لماذا رفع النموذج درجة هذا الطلب؟ | Local explanation
نختار أعلى درجة خام داخل عينة SHAP لتوضيح حالة مرتفعة الدرجة؛ لا يعتمد الاختيار على نتيجتها الفعلية. تعرض waterfall أكبر الإسهامات مع تجميع البقية، وتبقى جميع الإسهامات محفوظة في الملف. ثلاث عبارات سبب كحد أقصى تُشتق من الإسهامات الموجبة الفعلية، لا من قائمة أسباب جاهزة.

قيمة الخاصية المستخدمة قد تكون معوضة؛ ارجع إلى المدخل الأصلي قبل تفسيرها. لا تعني مساهمة موجبة أن الخاصية مرتفعة دائمًا، ولا تعني أن تغييرها سيغير نتيجة الشخص. نفحص تغير درجة bureau_score بمقدار ±1 كاختبار محلي صغير، وليس ضمان استقرار لكل الطلبات أو الخصائص.

In [ ]:
local_index = int(np.argmax(shap_arrays["raw_margin"]))
local_id = str(shap_arrays["application_ids"][local_index])
case = parts["evaluation"].set_index("application_id", drop=False).loc[[local_id]]
descriptions = {f["name"]: f["description_ar"] for f in contract["features"]}
reasons = reason_codes(shap_arrays["values"][local_index], shap_arrays["feature_names"],
    shap_arrays["data"][local_index], descriptions)
reasons["application_id"] = local_id
reasons["was_imputed"] = [bool(case.iloc[0][feature] != case.iloc[0][feature]) for feature in reasons.feature]
reasons["source"] = EXPLANATION_SOURCE
raw_local = float(expit(shap_arrays["raw_margin"][local_index]))
cal_local = float(map_probability(raw_local, mapping))
print(f"{local_id} | raw score={raw_local:.4f} | calibrated probability={cal_local:.4f}")
display(reasons[["feature", "value_used", "was_imputed", "contribution_log_odds", "reason_ar"]])
plt.figure()
shap.plots.waterfall(explanation[local_index], max_display=10, show=False)
plt.title(f"{EXPLANATION_SOURCE} · {local_id} · raw log-odds", fontsize=12, pad=45)
plt.savefig(ARTIFACTS / "shap_waterfall.png", bbox_inches="tight"); plt.show()
if EXPLANATION_SOURCE == "LIVE":
    local_checks = local_stability(model, case, contract, mapping)
    local_checks["source"] = "LIVE"
    display(local_checks)
else:
    local_checks = pd.DataFrame(columns=["perturbation", "bureau_score_used", "raw_probability", "calibrated_probability",
        "top_positive_features", "shared_top_reasons", "reference_reasons", "source"])
    print("Local perturbation check skipped in example mode; do not claim it ran.")

## 6. هل تتحسن الاحتمالات على فترة منفصلة؟ | Calibration
تعلم sigmoid من calibration فقط، دون أوزان فئات جديدة. يثبت النموذج والمعاير ثم تقاس النتائج على evaluation. نقارن AP وROC-AUC للترتيب، وBrier وlog-loss وECE لجودة الاحتمال. انخفاض Brier لا يثبت المعايرة المثالية وحدها.

ECE هنا مجموع **عدد الصفوف في الحاوية ÷ جميع الصفوف × |متوسط الاحتمال − تكرار الحدث|**، باستخدام عشر حاويات متساوية العرض. تُحفظ الحاويات الفارغة بقيم غير متاحة، ويظهر عدد كل حاوية. يتأثر ECE بالحدود وحجم العينة؛ لا تخفِ الحاويات القليلة.

تحت القطر يعني أن احتمال الحدث أعلى من تكراره المرصود؛ لا يعني الثقة الإحصائية. التحويل المتزايد للنموذج نفسه يحافظ على الترتيب هنا، لكن المعايرة لا تضمن تحسنًا، ولا يضمن ensemble من معايرات متعددة حفظ الترتيب. isotonic توسع اختياري يحتاج حذرًا مع 40 موجبًا فقط في عينة المعايرة المنشورة.

In [ ]:
calibration_metrics, bin_tables = {}, []
for label, column in [("raw", "raw_probability"), ("sigmoid", "calibrated_probability")]:
    calibration_metrics[label] = probability_metrics(evaluation[TARGET], evaluation[column])
    table = reliability_table(evaluation[TARGET], evaluation[column]); table["variant"] = label
    bin_tables.append(table)
reliability = pd.concat(bin_tables, ignore_index=True)
display(pd.DataFrame(calibration_metrics).T[["rows", "positives", "roc_auc", "average_precision", "brier", "ece", "log_loss"]].round(5))
brier_delta = calibration_metrics["sigmoid"]["brier"] - calibration_metrics["raw"]["brier"]
ece_delta = calibration_metrics["sigmoid"]["ece"] - calibration_metrics["raw"]["ece"]
print(f"Measured change (after - before): Brier {brier_delta:+.5f}, ECE {ece_delta:+.5f}")
fig, axes = plt.subplots(2, 1, figsize=(10, 8), layout="constrained", gridspec_kw={"height_ratios": [2, 1]})
for label, color, marker in [("raw", "#b45a18", "o"), ("sigmoid", "#137d72", "s")]:
    bins = reliability[reliability.variant == label]; nonempty = bins[bins['count'] > 0]
    axes[0].plot(nonempty.mean_probability, nonempty.observed_rate, marker=marker, color=color, label=label)
    for row in nonempty.itertuples():
        axes[0].annotate(str(row.count), (row.mean_probability, row.observed_rate), xytext=(4, 5 if label == "raw" else -13), textcoords="offset points", fontsize=8)
    positions = bins['bin'] + (-.2 if label == "raw" else .2)
    axes[1].bar(positions, bins['count'], width=.4, color=color, label=label)
axes[0].plot([0, 1], [0, 1], linestyle="--", color="#777", label="Agreement line")
axes[0].set(xlim=(0, 1), ylim=(0, 1), xlabel="Mean predicted probability", ylabel="Observed event rate")
axes[0].legend(); axes[0].grid(alpha=.2)
axes[1].set(xticks=np.arange(10), xticklabels=[f"{i/10:.1f}–{(i+1)/10:.1f}" for i in range(10)], ylabel="Requests", xlabel="Fixed-width probability bins")
axes[1].legend()
fig.suptitle(f"LIVE · {len(evaluation):,} evaluation requests · counts shown; sparse bins are noisy", fontsize=12)
fig.savefig(ARTIFACTS / "reliability_curve.png", bbox_inches="tight"); plt.show()

## 7. كم تتغير النتيجة؟ | Stability
نستخدم bootstrap مزدوجًا قبل/بعد المعايرة، مع سحب **العملاء** وإبقاء طلبات العميل معًا. نعرض الفترة المئينية 95% لـ AP ولتغير Brier. النموذج والمعاير ثابتان في التكرارات؛ لا تشمل الفترة عدم يقين التدريب أو تعلم المعايرة أو تغير الزمن المستقبلي. لا تمثل فترة ثقة لاحتمال طلب واحد.

نقارن ربعي التقييم أيضًا. تشتتهما وصف لاختلاف الفترات، وليس انحراف ثلاث طيات مستقلة أو فترة ثقة. إن تغيّرت النتائج بعد الاطلاع على التقييم، يلزم دليل جديد لتقييم ذلك التغيير.

In [ ]:
bootstrap, stability_summary = cluster_bootstrap(evaluation, config)
period_rows = []
for period, group in evaluation.groupby(pd.to_datetime(evaluation.application_date).dt.to_period("Q").astype(str)):
    for variant, column in [("raw", "raw_probability"), ("sigmoid", "calibrated_probability")]:
        period_rows.append({"period": period, "variant": variant, **probability_metrics(group[TARGET], group[column])})
period_metrics = pd.DataFrame(period_rows)
display(period_metrics[["period", "variant", "rows", "positives", "average_precision", "brier", "ece"]].round(5))
stability_summary["period_ap_sample_sd"] = {v: float(g.average_precision.std(ddof=1)) for v, g in period_metrics.groupby("variant")}
print("Paired customer-cluster 95% percentile intervals:")
display(pd.DataFrame({key: stability_summary[key] for key in ("raw_ap", "calibrated_ap", "brier_change")}).T)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
for i, key in enumerate(("raw_ap", "calibrated_ap")):
    interval = stability_summary[key]
    point = calibration_metrics["raw" if i == 0 else "sigmoid"]["average_precision"]
    if interval is not None:
        axes[0].hlines(i, interval["lower"], interval["upper"], color="#2b6b98", lw=3)
    axes[0].plot(point, i, "o", color="#2b6b98")
axes[0].set(yticks=[0, 1], yticklabels=["Raw AP", "Sigmoid AP"], xlabel="Average Precision", xlim=(0, 1), ylim=(-.6, 1.6))
axes[1].hist(bootstrap.brier_change, bins=20, color="#137d72", alpha=.85)
axes[1].axvline(0, color="#555", linestyle="--"); axes[1].set(xlabel="Brier after − before (negative = improvement)", ylabel="Bootstrap replicates")
fig.suptitle(f"LIVE · {stability_summary['valid']} customer-cluster replicates · fixed model and calibrator", fontsize=12)
fig.savefig(ARTIFACTS / "stability_summary.png", bbox_inches="tight"); plt.show()

## 8. هل تلائم منطقة المراجعة السعة؟ | Review zone
اخترنا العتبة على policy فقط بخسارة 10×FN+FP وسقف 12%. ننقلها عبر sigmoid المتزايد إلى مقياس الاحتمال الجديد؛ **لا نعيد اختيارها على evaluation، ولا ننسخ رقم عتبة اليوم الثالث لنموذج مختلف**. نفحص أن مجموعة الإشارات نفسها لم تتغير.

منطقة ±0.02 حول العتبة تشخيص للحالات القريبة؛ ليست تقدير عدم يقين فردي. إذا أردت مراجعتها مع جميع الإشارات، احسب **اتحاد المجموعتين** حتى لا تعد الطلب مرتين. قد يفشل السقف في فترة لاحقة حتى قبل إضافة المنطقة. يعرض التقرير ذلك كحاجة لمراجعة السعة؛ لا يرفع السقف ولا يخفي صفوفًا كي ينجح.

كل هذه إشارات محاكاة تعليمية؛ لا يوجد قرار موافقة أو رفض فعلي، ولا رسوم أو خصم درجات.

In [ ]:
review_flags, capacity_audit, review_band = review_audit(evaluation, chosen_policy["threshold"],
    calibrated_threshold, config.review_half_width)
policy_status = "CAPACITY_REVIEW_REQUIRED" if not capacity_audit.candidate_within_capacity.all() else "WITHIN_OBSERVED_CAPACITY"
display(pd.DataFrame([{**chosen_policy, "calibrated_threshold": calibrated_threshold, "selection_role": "policy"}]))
display(capacity_audit)
print(policy_status, "| Do not retune on evaluation.")
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout="constrained")
grid = np.linspace(0, 1, 300)
axes[0].plot(grid, map_probability(grid, mapping), color="#137d72")
axes[0].axhspan(review_band['lower'], review_band['upper'], color="#d7a354", alpha=.25, label="Diagnostic band ±0.02")
axes[0].plot(chosen_policy['threshold'], calibrated_threshold, 'o', color="#b45a18")
axes[0].set(xlim=(0, 1), ylim=(0, 1), xlabel="Raw model score", ylabel="Sigmoid probability")
axes[0].legend(loc="upper left", fontsize=9)
x = np.arange(len(capacity_audit))
for offset, column, label, color in [(-.25, 'capacity', '12% limit', '#aeb6bd'), (0, 'risk_flags', 'Risk flags', '#2b6b98'), (.25, 'candidate_review', 'Flags ∪ near threshold', '#b45a18')]:
    bars = axes[1].bar(x+offset, capacity_audit[column], width=.25, label=label, color=color)
    axes[1].bar_label(bars, padding=2, fontsize=9)
axes[1].set(xticks=x, xticklabels=capacity_audit.period, ylabel="Requests", ylim=(0, capacity_audit.candidate_review.max()*1.4))
axes[1].legend(fontsize=9)
fig.suptitle(f"LIVE · transported threshold {calibrated_threshold:.4f} · {policy_status}", fontsize=12)
fig.savefig(ARTIFACTS / "review_zone.png", bbox_inches="tight"); plt.show()

## 9. اكتب تقريرك | Your interpretation
استشهد بأرقامك ومقاماتها. اشرح الفرق بين العام والمحلي، ووحدة SHAP وخلفيته، وحدود السبب والترابط، ودليل المعايرة على الفترة المنفصلة، وما تشمله فترة bootstrap، وأثر منطقة المراجعة على السعة. اكتمال النصوص يعني جاهزًا للمراجعة فقط؛ لا يصحح صحتها ولا يمنح درجة.

In [ ]:
responses = {
    "global_local": "",
    "output_units": "",
    "reason_limits": "",
    "calibration_evidence": "",
    "stability": "",
    "review_capacity": "",
}
checkpoint = reflection_check(responses, EXPLANATION_SOURCE)
print(checkpoint["status"], "| Fields to complete:", checkpoint["missing"])

## 10. احفظ الأدلة والتقرير | Export
نزّل `day4_artifacts.zip` من Files → tamweel → artifacts، وافك الحزمة في جذر مشروعك للحفاظ على `artifacts/` و`reports/`. نزّل الدفتر واحفظه في `notebooks/04_explain_calibrate.ipynb`. تُحفظ ملفات JSON بدقة العتبة والمعايرة، وNPZ بقيم SHAP كاملة دون كائنات pickle.

عند إعادة قراءة الاحتمالات استخدم `pd.read_csv(path, float_precision="round_trip")`. نجاح التشغيل لا يعني نجاح سياسة السعة أو اكتمال تفسيرك. المثال المحفوظ يوسم التقرير كله غير صالح للتسليم كتجربتك؛ يُمنع مع `ASSESSMENT_MODE=1`.

In [ ]:
def json_safe(value):
    if isinstance(value, dict): return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, np.ndarray)): return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)): return int(value)
    if isinstance(value, (np.bool_,)): return bool(value)
    if isinstance(value, (float, np.floating)): return float(value) if np.isfinite(value) else None
    return value

technical_status = "TECHNICAL_READY" if EXPLANATION_SOURCE == "LIVE" else "EXAMPLE_ONLY_NOT_SUBMITTABLE"
calibration_payload = {"source": "LIVE", "evaluation_role": "evaluation", "metrics": calibration_metrics,
    "mapping": mapping, "policy_selection": chosen_policy, "calibrated_threshold": calibrated_threshold,
    "review_band": review_band, "capacity_status": policy_status, "brier_change": brier_delta, "ece_change": ece_delta,
    "prior_course_exposure": True, "not_a_final_untouched_test": True}
run = {"technical_status": technical_status, "learner_status": checkpoint["status"],
    "explanation_source": EXPLANATION_SOURCE, "model_calibration_source": "LIVE", "config": asdict(config),
    "assessment_mode": ASSESSMENT_MODE, "lab_revision": LAB_REVISION, "support_sha256": SUPPORT_HASHES,
    "previous_support": PREVIOUS_SUPPORT, "data_revision": COURSE_REVISION, "python": platform.python_version(),
    "elapsed_seconds": round(time.perf_counter()-RUN_STARTED, 2), "capacity_status": policy_status,
    "challenge_used_for_modeling": False, "no_automatic_grade": True}
csv_outputs = {"day4_roles.csv": assignments, "day4_predictions.csv": predictions,
    "permutation_importance.csv": permutation, "day4_shap_global.csv": shap_global.assign(source=EXPLANATION_SOURCE),
    "day4_reason_codes.csv": reasons, "day4_local_stability.csv": local_checks,
    "day4_reliability_bins.csv": reliability, "day4_period_metrics.csv": period_metrics,
    "day4_bootstrap.csv": bootstrap, "day4_policy_sweep.csv": policy_sweep,
    "day4_review_flags.csv": review_flags, "day4_capacity.csv": capacity_audit}
for filename, table in csv_outputs.items(): table.to_csv(ARTIFACTS / filename, index=False)
np.savez_compressed(ARTIFACTS / "shap_values_sample.npz", **shap_arrays)
model.named_steps["model"].booster_.save_model(str(ARTIFACTS / "day4_model.txt"))
json_outputs = {"calibration_metrics.json": calibration_payload, "day4_shap_metadata.json": shap_metadata,
    "day4_stability_summary.json": stability_summary, "day4_provenance.json": provenance,
    "day4_reflection.json": {"responses": responses, "checkpoint": checkpoint}, "day4_run.json": run}
for filename, payload in json_outputs.items():
    (ARTIFACTS / filename).write_text(json.dumps(json_safe(payload), ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
answer = lambda key: responses[key].strip() or "[اكتب تفسيرك من نتائجك هنا]"
status_text = "مثال تعليمي غير صالح للتسليم كتجربة شخصية" if EXPLANATION_SOURCE != "LIVE" else ("مسودة — أكمل تفسيرك" if checkpoint['missing'] else "جاهز للمراجعة؛ لا يعني اعتمادًا أو درجة")
reason_text = "\n".join(f"- {r.reason_ar} (SHAP={r.contribution_log_odds:+.4f} log-odds؛ قيمة معوضة: {r.was_imputed})" for r in reasons.itertuples())
capacity_text = "\n".join(f"- {r.period}: السقف {r.capacity}، الإشارات {r.risk_flags}، اتحاد المراجعة {r.candidate_review}." for r in capacity_audit.itertuples())
report = f"""# تقريرك: التفسير والمعايرة — Tamweel Lite

**الحالة:** {status_text}

**مصدر التفسير:** {EXPLANATION_SOURCE}. **النموذج والمعايرة:** LIVE. **السعة:** {policy_status}.

## النموذج والأدوار
LightGBM موزون، {config.trees} شجرة. الهدف حدث تعثر اصطناعي خلال90 يومًا بعد الطلب. الأدوار منفصلة زمنيًا وبالعملاء: تدريب {len(parts['fit'])}، معايرة {len(parts['calibration'])} ({int(parts['calibration'][TARGET].sum())} موجب)، سياسة {len(parts['policy'])}، تقييم {len(evaluation)}. الفجوات والتداخلات مستبعدة. سبق استخدام بيانات التقييم في الدورة، فهي ليست اختبارًا نهائيًا لم يمسّ.

## التفسير العام والمحلي
Permutation يقيس انخفاضAP على التقييم؛ إشارات المنطقة تُبدّل معًا. SHAP يفسر النموذج الخام بوحدةlog-odds وخلفية مسارات أشجار التدريب. base+sum(SHAP)=raw margin، ثمsigmoid للمجموع فقط. القيم ليست نقاط احتمال ولا تفسيرًا مباشرًا للنموذج المعاير.

{answer('global_local')}

{answer('output_units')}

الطلب الاصطناعي {local_id}: الدرجة الخام {raw_local:.5f} والاحتمال المعاير {cal_local:.5f}. اختير أعلى درجة داخل عينةSHAP دون استخدام النتيجة الفعلية.
{reason_text}

{answer('reason_limits')}

## دليل المعايرة
على {len(evaluation)} صفًا و{int(evaluation[TARGET].sum())} موجب: Brier {calibration_metrics['raw']['brier']:.6f} → {calibration_metrics['sigmoid']['brier']:.6f}؛ ECE {calibration_metrics['raw']['ece']:.6f} → {calibration_metrics['sigmoid']['ece']:.6f}. عشر حاويات متساوية العرض مع أعدادها فيday4_reliability_bins.csv. AP {calibration_metrics['raw']['average_precision']:.6f} → {calibration_metrics['sigmoid']['average_precision']:.6f}؛ ROC-AUC {calibration_metrics['raw']['roc_auc']:.6f} → {calibration_metrics['sigmoid']['roc_auc']:.6f}. هذه نتائج هذه العينة وليست ضمانًا لتحسن مستقبلي.

{answer('calibration_evidence')}

## الاستقرار
{stability_summary['valid']} تكرارbootstrap صالح بسحب العملاء؛ فترات مئينية95% مع تثبيت النموذج والمعاير. لا تشمل تعلم النموذج أو المعايرة أو الانجراف المستقبلي، ولا تصف احتمال فرد. انحرافAP بين ربعي التقييم وصفي فقط. اختبارbureau_score±1 {'نُفذ؛ راجع day4_local_stability.csv' if EXPLANATION_SOURCE == 'LIVE' else 'لم ينفذ في وضع المثال'}.

{answer('stability')}

## العتبة ومنطقة المراجعة
العتبة الخام {chosen_policy['threshold']!r} اختيرت علىpolicy بخسارة10×FN+FP وسقف12% ثم نُقلت إلى {calibrated_threshold!r}. لم تعدل باستخدام التقييم. المنطقة[{review_band['lower']:.5f}, {review_band['upper']:.5f}] تشخيصية بعرض±0.02 وليست فترة ثقة. الاتحاد يحسب الطلب مرة واحدة.
{capacity_text}

{answer('review_capacity')}

عند تجاوز السعة، وثّق الحاجة إلى تصميم سياسة جديدة على بيانات تطوير وتقييمها بدليل جديد. لا ترفع السقف ولا تقص الحالات بعد رؤية النتيجة. التفسير ليس سببية أو شهادة عدالة، والخسارة وحدات تعليمية لا رسوم أو خصم درجات. لا يستخدم هذا التمرين لتمويل حقيقي.
"""
(REPORTS / "INTERPRETABILITY_REPORT.md").write_text(report, encoding="utf-8")
figures = ["permutation_importance.png", "shap_beeswarm.png", "shap_waterfall.png", "reliability_curve.png", "stability_summary.png", "review_zone.png"]
artifact_files = list(csv_outputs) + list(json_outputs) + figures + ["environment.json", "shap_values_sample.npz", "day4_model.txt"]
with zipfile.ZipFile(ARTIFACTS / "day4_artifacts.zip", "w", zipfile.ZIP_DEFLATED) as bundle:
    for filename in artifact_files: bundle.write(ARTIFACTS / filename, "artifacts/" + filename)
    bundle.write(REPORTS / "INTERPRETABILITY_REPORT.md", "reports/INTERPRETABILITY_REPORT.md")
print(f"{technical_status} | {checkpoint['status']} | {EXPLANATION_SOURCE} | CPU")
print(f"Saved {len(artifact_files)+1} files in day4_artifacts.zip | {policy_status}")

## قبل أن تنتقل
افتح الرسوم وتحقق من الوحدات والمقامات. اشرح لماذا قد يحافظ تحويل واحد متزايد على الترتيب، ولماذا لا تثبت مساهمات SHAP السببية، ولماذا تحتاج منطقة المراجعة إلى حساب سعة مستقل. إذا ظهر مثال تعليمي، أعد التفسير على CPU لاحقًا؛ لا ترفعه كتجربتك.

[دليل اليوم الرابع](https://github.com/almiyead-rgb/sda-dsc-211-student-template/blob/main/DAY4_GUIDE.md) · [TreeExplainer ووحدات الخرج](https://shap.readthedocs.io/en/latest/generated/shap.TreeExplainer.html) · [المعايرة بنموذج مجمد](https://scikit-learn.org/1.6/modules/generated/sklearn.calibration.CalibratedClassifierCV.html) · [أهمية التبديل وحدود الترابط](https://scikit-learn.org/1.6/modules/permutation_importance.html)

Your report combines observed evidence with your reasoning. Next, compare ensemble choices and assemble a reproducible final delivery. Keep the final challenge separate.